In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import GeometryCollection
from shapely.ops import unary_union
import pygadm
import os

epsg = 'EPSG:4326' # https://en.wikipedia.org/wiki/World_Geodetic_System#WGS_84

# Use ISO 3166-1 alpha-3 code for the Netherlands
iso_code = 'NLD'
ad_level = 0

# Download the shapefile data
nl_gadm = pygadm.Items(admin=iso_code, content_level=ad_level).set_crs(epsg)

# load the shapefile data
nl_cbs = gpd.read_parquet('../administrative/cbs_nl.parquet').to_crs(epsg)
cbs = gpd.read_parquet('../administrative/cbs.parquet').to_crs(epsg)
gadm = gpd.read_parquet('../administrative/gadm.parquet').to_crs(epsg)

In [ ]:
def getUnclaimedArea(
    countryGdf: gpd.GeoDataFrame,
    municipalitiesGdf: gpd.GeoDataFrame
) -> gpd.GeoSeries:
    """
    Returns the area inside the country geometry that is not covered by any municipality.

    Args:
        countryGdf: Single polygon or multipolygon representing the country.
        municipalitiesGdf: Polygons representing all municipalities.

    Returns:
        GeoSeries of area(s) inside the country but not in any municipality.
    """
    countryUnion = countryGdf.union_all()
    municipalitiesUnion = municipalitiesGdf.union_all()
    uncoveredArea = countryUnion.difference(municipalitiesUnion)
    return gpd.GeoSeries([uncoveredArea], crs=countryGdf.crs)


def getOverclaimedArea(
    countryGdf: gpd.GeoDataFrame,
    municipalitiesGdf: gpd.GeoDataFrame
) -> gpd.GeoSeries:
    """
    Returns the area claimed by municipalities that lies outside the country's geometry.

    Args:
        countryGdf: Single polygon or multipolygon representing the country.
        municipalitiesGdf: Polygons representing all municipalities.

    Returns:
        GeoSeries of area(s) that municipalities claim but fall outside the country.
    """
    countryUnion = countryGdf.union_all()
    municipalitiesUnion = municipalitiesGdf.union_all()
    outsideArea = municipalitiesUnion.difference(countryUnion)
    return gpd.GeoSeries([outsideArea], crs=countryGdf.crs)


def getMultiplyClaimedArea(
    municipalitiesGdf: gpd.GeoDataFrame
) -> gpd.GeoSeries:
    """
    Returns the areas claimed by more than one municipality using spatial indexing.

    Args:
        municipalitiesGdf: GeoDataFrame of municipal boundaries.

    Returns:
        GeoSeries of overlapping geometries (polygonal only).
    """
    municipalitiesGdf = municipalitiesGdf.reset_index(drop=True)
    spatialIndex = municipalitiesGdf.sindex
    overlaps = []

    for i, geom in enumerate(municipalitiesGdf.geometry):
        if geom is None or geom.is_empty:
            continue

        # Get potential overlaps using bounding box intersection
        possibleMatchesIndex = list(spatialIndex.intersection(geom.bounds))
        possibleMatchesIndex.remove(i)  # Exclude self

        for j in possibleMatchesIndex:
            other = municipalitiesGdf.geometry[j]
            if other is None or other.is_empty:
                continue
            inter = geom.intersection(other)
            if not inter.is_empty and inter.geom_type in ('Polygon', 'MultiPolygon'):
                overlaps.append(inter)

    return gpd.GeoSeries(overlaps, crs=municipalitiesGdf.crs)

In [ ]:
def plotAreaDetails(
    country: gpd.GeoDataFrame,
    municipalities: gpd.GeoDataFrame,
    countryName: str = 'Netherlands',
    outputFile: str = 'details.pdf',
    figsize: tuple = (13, 8),
    verbose: bool = False,
    legendOutside: bool = True,
    crs: int = 28992,
    # Styling options
    nlColor: str = '#D9F0D3',
    nlAlpha: float = 0.8,
    nlEdgeColor: str = '#006400',
    nlLineWidth: float = 0.5,
    unassignedColor: str = '#1F78B4',
    unassignedAlpha: float = 0.6,
    overclaimedColor: str = '#084081',
    overclaimedAlpha: float = 1.0,
    multiplyClaimedColor: str = '#E31A1C',
    multiplyClaimedAlpha: float = 1.0,
    # Legend and output
    legendFontSize: int = 10,
    legendLoc: str = 'upper left',
    saveAsPDF: bool = True,
    saveAsPNG: bool = False,
    dpi: int = 300,
    # Geometry filtering
    minAreaQuantile: float = 0.2,
    # Edge styling for all non-country layers
    edgeStyle: dict | None = {'color': 'black', 'linewidth': 0.05}
) -> dict:
    """
    Visualizes and exports a map showing spatial inconsistencies in municipal claims
    within the given country, including unassigned areas, overclaimed regions, and
    multiply claimed zones.

    Args:
        country: GeoDataFrame representing the country's national boundary.
        municipalities: GeoDataFrame with municipal boundaries.
        countryName: Name of the country to be shown in legend labels.
        outputFile: Output filename (without extension if both formats are saved).
        figsize: Tuple controlling the figure size in inches.
        verbose: Whether to print information about empty or invalid geometries.
        legendOutside: Whether to place the legend outside the plot area.
        crs: EPSG code for projected CRS used in area computations and plotting.
        nlColor: Fill color for the country.
        nlAlpha: Opacity of the country's fill.
        nlEdgeColor: Edge color for the country.
        nlLineWidth: Edge line width for the country.
        unassignedColor: Fill color for unassigned areas.
        unassignedAlpha: Opacity for unassigned areas.
        overclaimedColor: Fill color for overclaimed areas.
        overclaimedAlpha: Opacity for overclaimed areas.
        multiplyClaimedColor: Fill color for multiply claimed areas.
        multiplyClaimedAlpha: Opacity for multiply claimed areas.
        legendFontSize: Font size used in the legend.
        legendLoc: Location string for the legend (e.g., 'upper left').
        saveAsPDF: Whether to save the figure as a PDF file.
        saveAsPNG: Whether to save the figure as a PNG file.
        dpi: Resolution for saving figures in dots per inch.
        minAreaQuantile: Threshold for excluding small geometries when zooming.
        edgeStyle: Dictionary defining edge color and width for all overlays (or None to disable).

    Returns:
        A dictionary containing:
            - 'countryAreaKm2': Total area of the country in square kilometers.
            - 'multiplyClaimedPct': Percentage of the area that is multiply claimed.
            - 'unassignedPct': Percentage of the area that is unassigned.
            - 'overclaimedPct': Percentage of the area that is overclaimed.
    """
    def hasNonzeroArea(gdf: gpd.GeoDataFrame) -> bool:
        return not gdf.empty and gdf.area.sum() > 0

    country_rd = country.to_crs(epsg=crs)
    countryArea = country_rd.area.sum()
    if countryArea == 0:
        raise ValueError('The country geometry has zero area after CRS transformation.')

    multiplyClaimed = getMultiplyClaimedArea(municipalities).to_crs(epsg=crs)
    unassigned = getUnclaimedArea(country, municipalities).to_crs(epsg=crs)
    overclaimed = getOverclaimedArea(country, municipalities).to_crs(epsg=crs)

    pcMultiply = multiplyClaimed.area.sum() / countryArea * 100 if hasNonzeroArea(multiplyClaimed) else 0.0
    pcUnassigned = unassigned.area.sum() / countryArea * 100 if hasNonzeroArea(unassigned) else 0.0
    pcOverclaimed = overclaimed.area.sum() / countryArea * 100 if hasNonzeroArea(overclaimed) else 0.0

    areaStr = f'{countryArea / 1_000_000:,.2f}'
    strMultiply = f'{pcMultiply:.2f}%'
    strUnassigned = f'{pcUnassigned:.2f}%'
    strOverclaimed = f'{pcOverclaimed:.2f}%'

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect('equal')

    threshold = country_rd.geometry.area.quantile(minAreaQuantile)
    mainArea = country_rd[country_rd.geometry.area > threshold]
    if mainArea.empty:
        if verbose:
            print('No geometries exceeded area threshold. Using full extent.')
        mainArea = country_rd

    minx, miny, maxx, maxy = mainArea.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    country_rd.plot(
        ax=ax,
        color=nlColor,
        alpha=nlAlpha,
        edgecolor=edgeStyle['color'] if edgeStyle else None,
        linewidth=edgeStyle['linewidth'] if edgeStyle else 0
    )

    if hasNonzeroArea(unassigned):
        unassigned.plot(
            ax=ax,
            color=unassignedColor,
            alpha=unassignedAlpha,
            edgecolor=edgeStyle['color'] if edgeStyle else None,
            linewidth=edgeStyle['linewidth'] if edgeStyle else 0
        )
    elif verbose:
        print('Unassigned area is empty or has zero area.')

    if hasNonzeroArea(overclaimed):
        overclaimed.plot(
            ax=ax,
            color=overclaimedColor,
            alpha=overclaimedAlpha,
            edgecolor=edgeStyle['color'] if edgeStyle else None,
            linewidth=edgeStyle['linewidth'] if edgeStyle else 0
        )
    elif verbose:
        print('Overclaimed area is empty or has zero area.')

    if hasNonzeroArea(multiplyClaimed):
        multiplyClaimed.plot(
            ax=ax,
            color=multiplyClaimedColor,
            alpha=multiplyClaimedAlpha,
            edgecolor=edgeStyle['color'] if edgeStyle else None,
            linewidth=edgeStyle['linewidth'] if edgeStyle else 0
        )
    elif verbose:
        print('Multiply claimed area is empty or has zero area.')

    legendHandles = [
        Patch(facecolor=nlColor, alpha=nlAlpha, label=f'{countryName} {areaStr}km²'),
        Patch(facecolor=unassignedColor, alpha=unassignedAlpha, label=f'{strUnassigned} unassigned'),
        Patch(facecolor=overclaimedColor, alpha=overclaimedAlpha, label=f'{strOverclaimed} abroad'),
        Patch(facecolor=multiplyClaimedColor, alpha=multiplyClaimedAlpha, label=f'{strMultiply} disputed'),
    ]

    ax.axis(False)

    if legendOutside:
        plt.legend(
            handles=legendHandles,
            loc=legendLoc,
            fontsize=legendFontSize,
            frameon=False,
            bbox_to_anchor=(1.05, 1),
            borderaxespad=0
        )
    else:
        plt.legend(
            handles=legendHandles,
            loc=legendLoc,
            fontsize=legendFontSize,
            frameon=False
        )

    base, _ = os.path.splitext(outputFile)
    if saveAsPDF:
        plt.savefig(base + '.pdf', format='pdf', bbox_inches="tight", dpi=dpi)
    if saveAsPNG:
        plt.savefig(base + '.png', format='png', bbox_inches="tight", dpi=dpi)

    plt.show()

    return {
        'area': countryArea / 1_000_000,
        'disputed': pcMultiply,
        'abandoned': pcUnassigned,
        'abroad': pcOverclaimed,
    }


In [ ]:
res_gadm = plotAreaDetails(nl_gadm, gadm, legendOutside=False, edgeStyle=None, saveAsPDF=True, saveAsPNG=False, outputFile='nl_gadm.pdf')

In [ ]:
res_gadm_cbs = plotAreaDetails(nl_gadm, cbs, legendOutside=False, edgeStyle=None, saveAsPDF=True, saveAsPNG=False, outputFile='nl_gadm_cbs.pdf')

In [ ]:
res_cbs = plotAreaDetails(nl_cbs, cbs, legendOutside=False, edgeStyle=None, saveAsPDF=True, saveAsPNG=False, outputFile='nl_cbs_cbs.pdf')

In [ ]:
res_cbs_gadm = plotAreaDetails(nl_cbs, gadm, legendOutside=False, edgeStyle=None, saveAsPDF=True, saveAsPNG=False, outputFile='nl_cbs_gadm.pdf')

In [ ]:
import pandas as pd, pyperclip as pc 

pc.copy(
    pd.DataFrame(
        {'GADM': res_gadm, 'GADM and CBS': res_gadm_cbs, 'CBS': res_cbs, 'CBS and GADM': res_cbs_gadm}
    ).map(lambda x: f'{x:,.2f}' if isinstance(x, float) else x).to_latex(
        index=True, escape=False, column_format='lrrrr'
    )
)


In [ ]:
def findRealOverlaps(
    gdf: gpd.GeoDataFrame,
    minOverlapArea: float = 1e4,
    nameColumn: str = 'Municipality',
    verbose: bool = False
) -> gpd.GeoDataFrame:
    """
    Find significant overlaps between nearby geometries in a GeoDataFrame.

    Parameters:
    - gdf: GeoDataFrame containing named polygon geometries.
    - minOverlapArea: Minimum overlap area in square meters to retain.
    - nameColumn: Name of the column in gdf to use for municipality labels.
    - verbose: If True, prints number of overlaps found.

    Returns:
    - GeoDataFrame with columns: index_1, index_2, muni_1, muni_2, geometry, area
    """
    gdf = gdf.to_crs('EPSG:28992').copy()
    _ = gdf.sindex  # force spatial index creation

    # Spatial join to find intersecting geometries
    candidates = sjoin(gdf, gdf, how='inner', predicate='intersects')
    candidates['index_1'] = candidates.index
    candidates['index_2'] = candidates['index_right']

    # Remove self-matches and symmetric duplicates
    candidates = candidates[candidates['index_1'] != candidates['index_2']]
    candidates['pair'] = candidates.apply(
        lambda row: tuple(sorted((row['index_1'], row['index_2']))), axis=1
    )
    candidates = candidates.drop_duplicates(subset='pair')

    geoms = gdf.geometry
    names = gdf[nameColumn]

    rows = []
    for idx1, idx2 in candidates['pair']:
        geom1 = geoms.loc[idx1]
        geom2 = geoms.loc[idx2]
        inter = geom1.intersection(geom2)
        if not inter.is_empty and inter.area > minOverlapArea:
            rows.append({
                'index_1': idx1,
                'index_2': idx2,
                'muni_1': names.loc[idx1],
                'muni_2': names.loc[idx2],
                'geometry': inter,
                'area': inter.area
            })

    if verbose:
        print(f'Found {len(rows)} valid overlaps.')

    if rows:
        return gpd.GeoDataFrame(rows, crs='EPSG:28992', geometry='geometry')
    else:
        return gpd.GeoDataFrame(columns=['index_1', 'index_2', 'muni_1', 'muni_2', 'geometry', 'area'],
                                geometry='geometry', crs='EPSG:28992')


In [ ]:
from shapely.ops import unary_union
from geopandas.tools import sjoin

def summarizeOverlapAreas(overlaps: gpd.GeoDataFrame) -> tuple[float, float]:
    """
    Compute the total area of overlaps in two ways:
    - Sum of all pairwise overlaps (may double-count shared regions)
    - Total unique area covered by all overlaps (no double-counting)

    Parameters:
    - overlaps: GeoDataFrame from findRealOverlaps, must have 'geometry' and 'area' columns.

    Returns:
    - Tuple: (sumOfPairwiseAreas, totalUniqueOverlapArea)
    """
    if overlaps.empty:
        return 0.0, 0.0

    # 1. Sum of individual intersection areas (possibly with overcounting)
    sumOfPairwiseAreas = overlaps['area'].sum()

    # 2. Union of all overlap geometries to avoid double/triple counting
    unioned = unary_union(overlaps.geometry.values)
    totalUniqueOverlapArea = unioned.area

    return sumOfPairwiseAreas, totalUniqueOverlapArea


In [ ]:
overlaps = findRealOverlaps(gadm, nameColumn='Municipality', minOverlapArea=0, verbose=True)
sumPairwise, sumUnique = summarizeOverlapAreas(overlaps)

print(f'Sum of pairwise overlaps (may overcount): {sumPairwise:,.0f} m²')
print(f'Total unique overlap area (no overcounting): {sumUnique:,.0f} m²')
print(f'Percentage of total area: {sumUnique / gadm.to_crs(28992).geometry.area.sum() * 100:.2f}%')

# Quick look
overlaps[['muni_1', 'muni_2', 'area']].sort_values('area', ascending=False).round(2)

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import Polygon

def findTripleOverlaps(overlaps: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Identify areas where 3 or more municipalities overlap.

    Parameters:
    - overlaps: GeoDataFrame with pairwise overlaps (from findRealOverlaps)

    Returns:
    - GeoDataFrame with geometry of triple (or more) overlaps and list of municipalities involved
    """
    if overlaps.empty:
        return gpd.GeoDataFrame(columns=['municipalities', 'geometry'], geometry='geometry', crs=overlaps.crs)

    # Build list of all municipalities and their intersection geometry
    records = []
    for _, row in overlaps.iterrows():
        records.append({'geometry': row['geometry'], 'municipalities': set([row['muni_1'], row['muni_2']])})
    
    # Initialize with first
    tripleZones = []

    for i, rec1 in enumerate(records):
        for j in range(i + 1, len(records)):
            rec2 = records[j]
            inter = rec1['geometry'].intersection(rec2['geometry'])
            if not inter.is_empty and isinstance(inter, Polygon) and inter.area > 0:
                combinedMuni = rec1['municipalities'].union(rec2['municipalities'])
                if len(combinedMuni) >= 3:
                    tripleZones.append({'municipalities': combinedMuni, 'geometry': inter})
    
    if tripleZones:
        return gpd.GeoDataFrame(tripleZones, geometry='geometry', crs=overlaps.crs)
    else:
        return gpd.GeoDataFrame(columns=['municipalities', 'geometry'], geometry='geometry', crs=overlaps.crs)


In [ ]:
tripleOverlaps = findTripleOverlaps(overlaps)

print(f'Found {len(tripleOverlaps)} triple-or-more overlaps.')

# Optionally inspect:
tripleOverlaps[['municipalities', 'geometry']]

In [ ]:
these = set(overlaps[overlaps.muni_1.str.contains('Capelle')].muni_2.values) | set(overlaps[overlaps.muni_1.str.contains('Capelle')].muni_1.values)

### Finding the Point Closest to Multiple Geometries

The function `findClosestPointToGeometries` computes the point in 2D space that minimizes the **sum of distances** to a given list of geometries (such as polygons or multipolygons). This can be interpreted as a generalization of the *geometric median* problem, extended to arbitrary shapes.

#### Problem Statement

Given a list of Shapely geometries `geoms`, we seek a point $p = (x, y)$ such that:

$$
\text{minimize } f(x, y) = \sum_{i=1}^n d(p, G_i)
$$

where:
- $d(p, G_i)$ is the shortest distance from point $p$ to geometry $G_i$,
- $G_i \in \text{geoms}$ for $i = 1, \dots, n$.

#### Method

1. **Objective Function**: Takes a 2D point and returns the sum of distances to all geometries.
2. **Initial Guess**: Uses the average of the bounding boxes’ centers as the starting point.
3. **Optimization**: Uses `scipy.optimize.minimize` (L-BFGS-B) to minimize the objective function.
4. **Result**: Returns a Shapely `Point` if successful; `None` otherwise.

#### Our Use Case

- We will center our explorations on such points that are close to all geometries.

#### Notes

- Geometries should be in a **projected CRS** (e.g. EPSG:28992 or EPSG:3857) for distance accuracy.
- Invalid geometries may cause optimization to fail.
- This minimizes **sum of distances**, not the distance to centroids or containment.


In [ ]:
from shapely.geometry import Point, base
import numpy as np
from scipy.optimize import minimize

def pointToShapeDistance(xy: np.ndarray, shapes: list[base.BaseGeometry]) -> float:
    """Objective function: sum of distances from (x, y) to each shape."""
    p = Point(xy)
    return sum(p.distance(shape) for shape in shapes)

def findClosestPointToGeometries(geoms: list[base.BaseGeometry]) -> Point:
    """Finds the point minimizing the sum of distances to all geometries."""
    if not geoms:
        raise ValueError('Empty list of geometries')

    # Check if all geometries are valid
    if not all(g.is_valid for g in geoms):
        print('[Warning] One or more geometries are invalid.')

    # Use centroid of bounding box of all shapes as initial guess
    all_bounds = np.array([geom.bounds for geom in geoms])
    center_x = all_bounds[:, [0, 2]].mean()
    center_y = all_bounds[:, [1, 3]].mean()
    x0 = np.array([center_x, center_y])

    result = minimize(pointToShapeDistance, x0, args=(geoms,), method='L-BFGS-B')
    
    if result.success:
        return Point(result.x)
    else:
        print('[Optimization failed]')
        print('Message:', result.message)
        print('Last evaluated point:', result.x)
        print('Final objective value:', result.fun)
        return None


In [ ]:
municipalities_to_show = these - {'Rotterdam'}
colors_to_use = { m : c for m, c in zip(municipalities_to_show, ['blue', 'green', 'orange'])}
colors_to_use

In [ ]:
nl_of_interest = gadm[gadm.Municipality.isin(municipalities_to_show)].copy()
nl_of_interest['color'] = nl_of_interest.Municipality.map(colors_to_use)

In [ ]:
geoms_to_use = nl_of_interest.geometry.values.tolist()
closest_point = findClosestPointToGeometries(geoms_to_use)

In [ ]:
import os
import wget

url = 'https://raw.githubusercontent.com/gromicho/tools/main/jg_folium.py'
filename = 'jg_folium.py'

# Ensure the file is removed before downloading
if os.path.exists(filename):
    os.remove(filename)

# Download the file (this will now always download the latest version)
wget.download(url, filename)

import jg_folium as jg

In [ ]:
m = nl_of_interest.explore(
    location=[closest_point.y-.001, closest_point.x],
    color=nl_of_interest['color'],  # outline color
    style_kwds={'fillOpacity': 0.35},
    zoom_start=14
)
jg.FoliumToPng(m, 'overlaps')
m

In [ ]:
perfectmatch = cbs[cbs.Municipality.isin(municipalities_to_show)].copy()
perfectmatch['color'] = perfectmatch.Municipality.map(colors_to_use)
m = perfectmatch.explore(
    location=[closest_point.y-.001, closest_point.x],
    color=perfectmatch['color'],  # outline color
    style_kwds={'fillOpacity': 0.35},
    zoom_start=14
)
jg.FoliumToPng(m, 'no_overlaps')
m

In [ ]:
overlaps['area_km2'] = overlaps.area / 1e6
overlaps.sort_values('area_km2',ascending=False)
# Separate out individual parts from MultiPolygons and GeometryCollections
overlapsClean = overlaps.explode(index_parts=False, ignore_index=True)
overlapsClean.sort_values('area_km2',ascending=False)

In [ ]:
from shapely.geometry import Point, Polygon, MultiPolygon, LineString

overlapsClean = overlapsClean[
    overlapsClean.geometry.apply(lambda geom: isinstance(geom, (Polygon, MultiPolygon)))
]
overlapsClean.sort_values('area_km2',ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(30, 30))
gadm.to_crs(overlapsClean.crs).plot(ax=ax, color='lightgrey', alpha=.5, edgecolor='grey', linewidth=0.1)
overlapsClean.plot(ax=ax, color='yellow', edgecolor='red', alpha=0.7,linewidth=0.8)
# plt.title('Cleaned and Filtered Overlaps')
plt.axis('off')
plt.savefig('overlaps_cleaned.pdf', dpi=300, bbox_inches='tight')
plt.show()